# ConformalGuard on R-Judge

Real-data experiment: conformal false-block control and scorer comparison on the 571 human-labelled R-Judge records.

**Before running:** Runtime → Change runtime type → GPU. Run the cells top to bottom. The last cell downloads `rjudge_results.zip`; upload that file back to the assistant.

In [ ]:
!nvidia-smi -L
!git clone --depth 1 -b claude/implement-conformalguard-UGndd https://github.com/rogerpanel/CV.git
!git clone --depth 1 https://github.com/Lordog/R-Judge.git
!pip -q install sentence-transformers scikit-learn pytest
%cd CV/ConformalGuard

## 1. Sanity check
Unit tests for the conformal layer and the R-Judge code (about one minute). All must pass before continuing.

In [ ]:
!python -m pytest tests/test_core.py tests/test_adaptive.py tests/test_rjudge.py -q

## 2. Main experiment
100 random splits, 100 prompt-grouped splits and 10 splits per held-out category, with a frozen sentence encoder for node features. Progress is printed per split. If the session is short on time, lower `--reps` to 50.

In [ ]:
!python scripts/rjudge_run.py --rjudge-dir ../../R-Judge --features st \
    --st-model sentence-transformers/all-MiniLM-L6-v2 \
    --reps 100 --loco-reps 10 --out runs/rjudge_st_minilm

## 3. Same experiment with a stronger text encoder (optional)
Shows whether the conclusions depend on the feature extractor.

In [ ]:
!python scripts/rjudge_run.py --rjudge-dir ../../R-Judge --features st \
    --st-model BAAI/bge-base-en-v1.5 \
    --reps 100 --loco-reps 10 --out runs/rjudge_st_bge

## 4. LLM-judge baseline (optional, uses the GPU)
R-Judge's own two-stage judging protocol with an open chat model. On a 16 GB GPU (T4) add `--load-4bit` after running `pip -q install bitsandbytes`. Try `--limit 10` first to see the speed. An interrupted run resumes where it stopped.

In [ ]:
!pip -q install accelerate
!python scripts/rjudge_llm_judge.py --rjudge-dir ../../R-Judge \
    --model Qwen/Qwen2.5-7B-Instruct --out runs/rjudge_judge_qwen7b

## 5. Package the results
Downloads one zip with every `results.json`, `summary.md` and the judge predictions.

In [ ]:
!zip -qr rjudge_results.zip runs -x '*/cache/*'
!ls -la rjudge_results.zip
from google.colab import files
files.download('rjudge_results.zip')